# Network Intrusion Detection: v7 (two-stage model, tuned stage 2)

**What the data showed**
- Train is 94.5% Availability, almost all of it one SYN-flood signature on port 21 (one 120-byte forward packet, fixed headers/window size).
- Train and test are almost perfectly separable (adversarial AUC ~ 1.0). The shift is in **timing / rate / response** features.
- On port 21, test rows have the **same forward-packet signature** as train but different timing/response behavior.
- Timing/rate features are still *needed* to separate Benign / Reconnaissance / Spoofing (public score: all features 0.845, no rate features 0.822, no timing at all 0.729).

**The model**
1. **Stage 1: Availability vs not**, using only stable forward-packet features (no ports, timing, rate or response features).
2. **Stage 2: Benign vs Reconnaissance vs Spoofing**, using all features, as an average of 3 tuned LightGBM settings.

**Validation:** rows are clustered within each class and whole clusters are held out (`StratifiedGroupKFold` for stage 1; a seed-dependent cluster-to-fold assignment for stage 2).

The notebook ends by writing `submission_v7_tuned.csv`. Runtime: about 5 minutes (about 3 more minutes for the optional validation cell).

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SEED = 42
TARGET = "Target"
ID = "id"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T0 = time.time()

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./{name}") + glob.glob(f"./data/{name}") + glob.glob(f"../data/{name}") + glob.glob(f"../../data/{name}")
    if not hits:
        raise FileNotFoundError(f"{name} not found. Add the competition data to the notebook.")
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)

In [ ]:
classes = sorted(train[TARGET].unique())            # Availability, Benign, Reconnaissance, Spoofing
cls2i = {c: i for i, c in enumerate(classes)}
K = len(classes)
y = train[TARGET].map(cls2i).values
train_mix = train[TARGET].value_counts(normalize=True).reindex(classes)
print(cls2i)
print(train[TARGET].value_counts())

AV = cls2i["Availability"]
is_av = (y == AV)
rest_classes = [c for c in classes if c != "Availability"]
K3 = len(rest_classes)
y3 = np.full(len(y), -1)
for j, c in enumerate(rest_classes):
    y3[y == cls2i[c]] = j
nonav_idx = np.where(~is_av)[0]

def prepare(train, test, drop=()):
    """Same feature matrix for train and test; categoricals share one category set; inf -> NaN."""
    feats = [c for c in train.columns if c not in (ID, TARGET) and c not in drop]
    full = pd.concat([train[feats], test[feats]], ignore_index=True)
    full = full.replace([np.inf, -np.inf], np.nan)
    for c in full.select_dtypes(exclude="number").columns:
        full[c] = full[c].astype("category")
    return full.iloc[:len(train)].reset_index(drop=True), full.iloc[len(train):].reset_index(drop=True)

X_tr, X_te = prepare(train, test)
PORTS = [c for c in ["id.orig_p", "id.resp_p"] if c in X_tr.columns]
num_cols = X_tr.select_dtypes("number").columns.tolist()
beh_cols = [c for c in num_cols if c not in PORTS]
print("features:", X_tr.shape[1], "| ports:", PORTS)

## 1. Pseudo-groups and shift-aware folds

Within each class, cluster rows on behavior features (all numeric except ports; log-scaled, standardized, PCA). Each cluster stands in for a "behavior profile", and `StratifiedGroupKFold` keeps each cluster inside one fold.

In [ ]:
Z = X_tr[beh_cols].copy()
Z = Z.fillna(Z.median())
Z = np.sign(Z) * np.log1p(Z.abs())
Z = StandardScaler().fit_transform(Z)
Z = PCA(n_components=min(20, Z.shape[1]), random_state=SEED).fit_transform(Z)

groups = np.zeros(len(y), dtype=int)
offset = 0
for ci, cls in enumerate(classes):
    idx = np.where(y == ci)[0]
    k = int(np.clip(len(idx) // 1500, 6, 20))
    km = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=4096).fit(Z[idx])
    groups[idx] = km.labels_ + offset
    offset += k

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y, groups))
for i, (tr, va) in enumerate(folds):
    print(f"fold {i}: val class counts {np.bincount(y[va], minlength=K)} | classes in train part: {len(np.unique(y[tr]))}/{K}")

## 2. Feature sets and helpers

- **Time / rate:** duration, `*_pkts_per_sec`, `payload_bytes_per_second`, `*_iat.*`, `active.*`, `idle.*`, bulk rates.
- **Response-side:** `bwd_*`, `down_up_ratio`, RST and ACK flag counts.

In [ ]:
A = list(X_tr.columns)
B = [c for c in A if c not in PORTS]
TIME_RATE = [c for c in B if ("per_sec" in c or "per_second" in c or "_iat." in c or c.startswith(("active.", "idle."))
                              or c == "flow_duration" or "bulk_rate" in c)]
RESPONSE = [c for c in B if c.startswith("bwd_") or c in ("down_up_ratio", "flow_RST_flag_count", "flow_ACK_flag_count")]
D = [c for c in B if c not in TIME_RATE]
E = [c for c in D if c not in RESPONSE]
F = D + PORTS
feature_sets = {"A": A, "B": B, "D": D, "E": E, "F": F}
desc = {"A": "all features", "B": "no ports", "D": "no ports, no timing/rate",
        "E": "no ports, no timing/rate/response", "F": "ports, no timing/rate"}
for k, f in feature_sets.items():
    print(f"  {k}: {desc[k]:40s} {len(f):3d} features")

lgb_params = dict(n_estimators=250, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0,
                  random_state=SEED, n_jobs=-1, verbose=-1)
# Stage 2 has few rows (about 4.6k), so use smaller leaves
lgb2_params = dict(lgb_params, n_estimators=300, min_child_samples=20)

def sample_w(y_tr, power, k):
    """power 0 = none, 0.5 = sqrt-balanced, 1 = fully balanced."""
    cnt = np.bincount(y_tr, minlength=k).astype(float)
    cnt[cnt == 0] = 1.0
    w = (len(y_tr) / (k * cnt)) ** power
    return w[y_tr]

def adv_auc(Xa, Xb, feats, n_est=100):
    """Label-free shift measure: can a model tell rows of Xa from rows of Xb using these features?"""
    X = pd.concat([Xa[feats], Xb[feats]], ignore_index=True)
    yy = np.r_[np.zeros(len(Xa)), np.ones(len(Xb))]
    oof = np.zeros(len(yy))
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, yy):
        m = lgb.LGBMClassifier(n_estimators=n_est, learning_rate=0.1, num_leaves=31, min_child_samples=50,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                               random_state=SEED, n_jobs=-1, verbose=-1).fit(X.iloc[tr], yy[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return roc_auc_score(yy, oof)

## 3. Stage 1: Availability vs everything else

Scored with the cluster-grouped CV. Pick: among sets within 0.005 of the best Availability F1, take the least shifted (lowest adversarial AUC between train and test).

In [ ]:
def run_stage1(feats):
    oof = np.zeros(len(y))
    for tr, va in folds:
        m = lgb.LGBMClassifier(**lgb_params).fit(X_tr.iloc[tr][feats], is_av[tr].astype(int))
        oof[va] = m.predict_proba(X_tr.iloc[va][feats])[:, 1]
    return oof

s1_oof, s1_rows = {}, []
for k, feats in feature_sets.items():
    s1_oof[k] = run_stage1(feats)
    pred = s1_oof[k] >= 0.5
    s1_rows.append(dict(set=k, desc=desc[k], n_features=len(feats),
                        avail_f1=f1_score(is_av, pred), precision=precision_score(is_av, pred),
                        recall=recall_score(is_av, pred), adv_auc=adv_auc(X_tr, X_te, feats)))
    print(f"set {k}: Availability F1 {s1_rows[-1]['avail_f1']:.4f} | adversarial AUC {s1_rows[-1]['adv_auc']:.4f}  [{time.time()-T0:.0f}s]")

s1 = pd.DataFrame(s1_rows).set_index("set").round(4)
best_f1 = s1["avail_f1"].max()
s1["eligible"] = s1["avail_f1"] >= best_f1 - 0.005
print()
print(s1)
s1_set = s1[s1["eligible"]]["adv_auc"].idxmin()
s1_feats = feature_sets[s1_set]
print("\nStage-1 feature set:", s1_set, "-", desc[s1_set])

# Fit stage 1 on all of train, score the test set
m1 = lgb.LGBMClassifier(**lgb_params).fit(X_tr[s1_feats], is_av.astype(int))
p_av_test = m1.predict_proba(X_te[s1_feats])[:, 1]
test_nonav = p_av_test < 0.5
print(f"Test rows flagged Availability by stage 1: {(~test_nonav).sum()} ({(~test_nonav).mean():.1%})")

## 5. Stage 2 (tuned): Benign vs Reconnaissance vs Spoofing

Trained on the 4,641 non-Availability training rows with all 82 features (ports included).
The three LightGBM settings below came from a random search (20 settings) scored with repeated cluster-grouped CV, then were checked on fresh splits the search never saw.
The final prediction averages the three models' probabilities.

In [ ]:
rest_idx = np.array([cls2i[c] for c in rest_classes])
Xs = X_tr.iloc[nonav_idx].reset_index(drop=True)      # non-Availability training rows
ys = y3[nonav_idx]
XsA = Xs[A].copy()
Xte_A = X_te[A].reset_index(drop=True)

base_cfg = (dict(lgb2_params), 0.5)                    # the v4 default (for comparison)
tuned_cfgs = [
    (dict(n_estimators=600, learning_rate=0.05, num_leaves=15, min_child_samples=80, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=20.0, max_depth=-1, random_state=SEED, n_jobs=-1, verbose=-1), 0.0),
    (dict(n_estimators=150, learning_rate=0.08, num_leaves=4, min_child_samples=5, subsample=0.6, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=5.0, max_depth=6, random_state=SEED, n_jobs=-1, verbose=-1), 0.0),
    (dict(n_estimators=400, learning_rate=0.03, num_leaves=4, min_child_samples=5, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=1.0, max_depth=4, random_state=SEED, n_jobs=-1, verbose=-1), 0.5),
]   # each entry: (LightGBM params, class-weight power)

def labels_for(Xtr_df, Xte_df, cfgs):
    P = np.zeros((len(Xte_df), K3))
    for params, power in cfgs:
        m = lgb.LGBMClassifier(**params).fit(Xtr_df, ys, sample_weight=sample_w(ys, power, K3))
        p = np.zeros((len(Xte_df), K3))
        p[:, m.classes_] = m.predict_proba(Xte_df)
        P += p / len(cfgs)
    out = np.where(p_av_test >= 0.5, AV, rest_idx[P.argmax(1)])     # stage 1 decides Availability
    return np.array(classes)[out]

lab_tuned = labels_for(XsA, Xte_A, tuned_cfgs)
lab_base = labels_for(XsA, Xte_A, [base_cfg])

sub = pd.DataFrame({ID: test[ID].values, TARGET: lab_tuned})
path = os.path.join(OUT, "submission_v7_tuned.csv")
sub.to_csv(path, index=False)
assert list(sub.columns) == list(sample.columns) == [ID, TARGET]
assert len(sub) == len(sample) and set(sub[ID]) == set(sample[ID])
assert sub[TARGET].isin(classes).all()
print("Saved", path)

mix = pd.DataFrame({"train": train_mix,
                    "test_pred_default_stage2": pd.Series(lab_base).value_counts(normalize=True),
                    "test_pred_tuned": pd.Series(lab_tuned).value_counts(normalize=True)}).reindex(classes).round(3)
print(mix)
print(f"\nTuned file differs from the default-stage-2 file on {int((lab_tuned != lab_base).sum())} of {len(sub)} test rows "
      f"({np.mean(lab_tuned != lab_base):.1%})")
print(f"Test rows flagged Availability by stage 1: {(p_av_test >= 0.5).sum()} ({(p_av_test >= 0.5).mean():.1%})")

## 6. Optional: validation of tuned vs default stage 2

Repeated cluster-grouped CV on the non-Availability rows. Both the clustering and the cluster-to-fold assignment depend on the seed, so each seed is a genuinely different split.
Seeds 10-14 were never used while choosing the settings. Expect the tuned average to be slightly ahead of the default (about +0.01), with large split-to-split variation (+/- 0.03), so treat the gain as small.
Set `RUN_VALIDATION = False` to skip (about 3 minutes).

In [ ]:
RUN_VALIDATION = True

if RUN_VALIDATION:
    Zs = Z[nonav_idx]     # behavior embedding built in the folds cell

    def make_folds(seed, k_per_class=15):
        rng_ = np.random.default_rng(seed)
        fold_of = np.zeros(len(ys), dtype=int)
        for c in range(K3):
            idx = np.where(ys == c)[0]
            k = int(min(k_per_class, max(5, len(idx) // 40)))
            km = MiniBatchKMeans(n_clusters=k, random_state=seed, n_init=3, batch_size=2048).fit(Zs[idx])
            order = rng_.permutation(k)
            cl_fold = {cl: pos % 5 for pos, cl in enumerate(order)}
            fold_of[idx] = [cl_fold[l] for l in km.labels_]
        return [(np.where(fold_of != f)[0], np.where(fold_of == f)[0]) for f in range(5)]

    def cv3(Xdf, cfgs, seeds):
        sc = []
        for sd in seeds:
            oof = np.zeros((len(ys), K3))
            for tr, va in make_folds(sd):
                P = np.zeros((len(va), K3))
                for params, power in cfgs:
                    m = lgb.LGBMClassifier(**params).fit(Xdf.iloc[tr], ys[tr], sample_weight=sample_w(ys[tr], power, K3))
                    p = np.zeros((len(va), K3))
                    p[:, m.classes_] = m.predict_proba(Xdf.iloc[va])
                    P += p / len(cfgs)
                oof[va] = P
            sc.append(f1_score(ys, oof.argmax(1), average="macro"))
        return np.array(sc)

    seeds = range(10, 15)
    s_base = cv3(XsA, [base_cfg], seeds)
    s_tuned = cv3(XsA, tuned_cfgs, seeds)
    print(f"default stage 2 : 3-class macro-F1 {s_base.mean():.4f} +/- {s_base.std():.4f}   per split {np.round(s_base, 3)}")
    print(f"tuned stage 2   : 3-class macro-F1 {s_tuned.mean():.4f} +/- {s_tuned.std():.4f}   per split {np.round(s_tuned, 3)}")
    print(f"tuned is better on {(s_tuned > s_base).sum()} of {len(s_base)} splits")

## What to record

- Public score of `submission_v7_tuned.csv` next to the earlier files (stage-2 all features 0.84538, no ports 0.84253, no rate features 0.82231, no timing 0.72928).
- At the end of the contest, use the **Select** boxes to choose your 2 final submissions. Prefer one robust pick and one close alternative rather than only the two highest public scores, because the private leaderboard uses different hidden profiles.